# Klassifikation mit Tensorflow/Keras

In diesem Notebook verwenden wir ein einfaches DNN für die Klassifikation. Wir nutzen den Datensatz zu Fahrzeugemissionen, den die Stadt Albuquerque [hier](https://www.cabq.gov/abq-data/) bereitstellt, um vorherzusagen, ob ein Fahrzeug den Abgastest nicht besteht. Ziel ist es, die Luftverschmutzung in Albuquerque zu verringern. Vieles übernehmen wir aus der Arbeit, die [hier](https://github.com/Guli-Y/wimlds_emissions) entstanden ist.

Da es ein großer Datensatz ist, haben wir eine Teilmenge der Daten genommen. Außerdem überspringen wir den Teil zur Exploration.

In [ ]:
# Alle benötigten Bibliotheken importieren
import numpy as np
import pandas as pd
from termcolor import colored
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, recall_score

import tensorflow as tf

np.random.seed(42)
tf.random.set_seed(42)

df = pd.read_csv("../data/vehicle_emissions.csv", index_col=0)

**Beschreibung der Variablen:**

|Variable|Definition   | Schlüssel  |  Typ |
|---|---|---|---|
| RESULT | Test bestanden? |   0 = Ja, 1 = Nein | Dichotom | 
| VIN | Fahrzeug-Identifizierungsnummer | Text | 17 Zeichen |
| TEST_SDATE | Datum/Uhrzeit des Testbeginns | MM/DD/YYYY HH:MM:SS | Datum/Uhrzeit |
| MODEL_YEAR | Modelljahr des Fahrzeugs |	Zahl | Jahr |
| GVWR | Zulässiges Gesamtgewicht | Zahl | Verhältnisskaliert |
| ENGINE_SIZE | Hubraum | Zahl | Verhältnisskaliert |
| ODOMETER | Tachostand, Anzahl der Meilen | Zahl | Verhältnisskaliert |
| MAKE | Fahrzeugmarke | Text | Nominal |

In [ ]:
df.head()

## Datenbereinigung

Ein Blick auf die Informationen zu jeder Spalte und auf die fehlenden Werte zeigt uns, dass wir unsere Daten bereinigen müssen, bevor wir sie für die Modellierung verwenden können.

In [ ]:
# TEST_SDATE in ein datetime-Objekt umwandeln
df["TEST_SDATE"] = pd.to_datetime(df["TEST_SDATE"])

Wir berechnen das Fahrzeugalter in Jahren näherungsweise, indem wir das Modelljahr vom Testjahr abziehen und 2 Jahre addieren. So vermeiden wir Fahrzeuge mit dem Alter null (z. B. Autos, die in ihrem Modelljahr getestet wurden), bevor wir die Laufleistung durch das Alter teilen.

In [ ]:
# VEHICLE_AGE erstellen
df["VEHICLE_AGE"] = (
    df.TEST_SDATE.dt.year.astype("int") - df.MODEL_YEAR.astype("int") + 2
)

Wir bereinigen die Spalte VIN und füllen GVWR-Werte, die fehlen oder null sind, wenn möglich auf; andernfalls entfernen wir sie.

In [ ]:
print(colored(f"\nRecords in input data: {df.shape[0]}", "red"))
# Median von GVWR für jede VIN berechnen
# VIN bereinigen
df["VIN"] = df["VIN"].astype("string").str.strip().str.lower()
# Median von GVWR berechnen
tmp = df[["VIN", "GVWR"]].groupby("VIN").agg({"GVWR": "median"})
tmp.reset_index(inplace=True)
# tmp mit df zusammenführen
df = df.merge(tmp, how="left", on="VIN", suffixes=("_0", ""))
# 0 durch np.nan ersetzen
df.loc[df.GVWR == 0, "GVWR"] = np.nan
df.loc[df.GVWR_0 == 0, "GVWR_0"] = np.nan
# Fehlende Werte in GVWR mit GVWR_0 auffüllen
df["GVWR"] = df.GVWR.fillna(df.GVWR_0)
# GVW behalten und GVWR_0 entfernen
df = df.drop(columns=["GVWR_0"])
# NA-Werte in GVWR entfernen
print("\nRecords with missing GVWR:", df.GVWR.isnull().sum())
df = df[~df.GVWR.isnull()]
# Niedrige Werte in GVWR entfernen
df = df[df.GVWR > 1000]
print(
    colored(
        f"\nRecords after droping rows where GVWR is < 1000 or missing: {df.shape[0]}",
        "red",
    )
)

Damit die Testergebnisse unabhängig sind, behalte nur das erste, wenn es innerhalb von 90 Tagen mehrere gibt.

In [ ]:
# Wenn ein Fahrzeug innerhalb von 90 Tagen mehrere Testeinträge hat, den frühesten Eintrag behalten
df = df.sort_values("TEST_SDATE")
df = df.loc[~(df.groupby("VIN")["TEST_SDATE"].diff() < np.timedelta64(90, "D"))]
print(
    colored(
        f"\nRecords after keeping only the earliest test within 90 days for each vehicle: {df.shape[0]}",
        "red",
    )
)

Jetzt können wir die Spalte `VIN` entfernen, da sie keine nützlichen Informationen enthält.

In [ ]:
df = df.drop(columns=["VIN"])

Bereinige die Spalte `ODOMETER`, indem du Nullen und die Werte `8888888` entfernst.

In [ ]:
# Nullen und 8888888 in ODOMETER entfernen
print("\nRecords where ODOMETER = 0:", df[df.ODOMETER == 0].shape[0])
df = df[(df.ODOMETER != 0) & (df.ODOMETER != 8888888)]
print(
    colored(
        f"\nRecords after dropping rows where ODOMETER is 0 or 8,888,888: {df.shape[0]}",
        "red",
    )
)

Wir berechnen die Meilen pro Jahr.

In [ ]:
# MILE_YEAR aus ODOMETER erstellen
df["MILE_YEAR"] = np.round(df["ODOMETER"] / df["VEHICLE_AGE"], 2)

Anhand der erstellten Features `MILE_YEAR` und `VEHICLE_AGE` entfernen wir eindeutige Ausreißer: Einträge mit über 40.000 Meilen pro Jahr und Fahrzeuge, die älter als 10 Jahre sind und unter 1.000 Meilen pro Jahr aufweisen.

In [ ]:
# Ausreißer entfernen
df = df[df.MILE_YEAR <= 40000]
df = df[~((df.VEHICLE_AGE > 10) & (df.MILE_YEAR < 1000))]
print(
    colored(
        f"\nRecords after dropping rows where MILE_YEAR > 40,000 or (VEHICLE_AGE > 10 and MILE_YEAR < 1,000): {df.shape[0]}",
        "red",
    )
)

Wir berechnen das Verhältnis von Hubraum zu Gewicht.

In [ ]:
# ENGINE_WEIGHT_RATIO erstellen
df["ENGINE_WEIGHT_RATIO"] = np.round(df["ENGINE_SIZE"] / df["GVWR"], 2)

Wir entfernen die verbleibenden NaN-Werte.

In [ ]:
df = df.dropna()
df = df.reset_index(drop=True)

Und wir wählen eine Teilmenge der Spalten aus, die am nützlichsten sind.

In [ ]:
cols = [
    "MODEL_YEAR",
    "VEHICLE_AGE",
    "MILE_YEAR",
    "MAKE",
    "ENGINE_WEIGHT_RATIO",
    "RESULT",
]
df = df[cols]

Damit die Anzahl der Spalten nach dem One-Hot Encoding nicht explodiert, benennen wir jede Marke, die bei weniger als 1% der Autos vorkommt, in `other` um.

In [ ]:
make_threshold = 0.01
# Ein Marken-Label 'other' für alle Marken erstellen, die <1% der Autos ausmachen
value_counts_norm = df["MAKE"].value_counts(normalize=True)
to_other = value_counts_norm[value_counts_norm < make_threshold]
# print(f"\n{len(to_other)} make labels each account for less than {round((make_threshold *100), 2)}% of cars and together account for {(round(to_other.sum(), 4)) *100}% of cars")
to_keep = value_counts_norm[value_counts_norm >= make_threshold]
makes_keep = list(to_keep.index)
makes_keep.sort()
other_makes = [make for make in df["MAKE"].unique() if make not in makes_keep]
df["MAKE"] = df["MAKE"].replace(other_makes, "other")
# Markenspalte per One-Hot Encoding kodieren
df = pd.get_dummies(df, drop_first=True, dtype=float)

Wir teilen unsere Daten in einen Trainings-, einen Validierungs- und einen Testdatensatz auf.

In [ ]:
# Eine Hilfsfunktion aus sklearn verwenden, um deinen Datensatz aufzuteilen und zu mischen.
train_df, test_df = train_test_split(df, test_size=0.2)
train_df, val_df = train_test_split(train_df, test_size=0.2)

# np-Arrays aus Labels und Features bilden.
train_labels = np.array(train_df.pop("RESULT"))
val_labels = np.array(val_df.pop("RESULT"))
test_labels = np.array(test_df.pop("RESULT"))

train_features = np.array(train_df)
val_features = np.array(val_df)
test_features = np.array(test_df)

print(f"Average class probability in training set:   {train_labels.mean():.4f}")
print(f"Average class probability in validation set: {val_labels.mean():.4f}")
print(f"Average class probability in test set:       {test_labels.mean():.4f}")

Wir sehen, dass wir es mit einem stark unbalancierten Datensatz zu tun haben. Um damit umzugehen, verwenden wir Klassengewichte.

Neuronale Netze sind sehr empfindlich gegenüber der Skalierung der Eingabedaten, und kleine Unterschiede in der Skalierung können sich erheblich auf die Leistung des Netzes auswirken. Wenn wir die Eingabedaten richtig skalieren, können wir sicherstellen, dass das neuronale Netz effektiv lernen und treffsichere Vorhersagen liefern kann. Deshalb fügen wir eine Normalisierungsschicht hinzu, die wir nur auf dem Trainingsdatensatz anpassen, um Data Leakage zu vermeiden. 

In [ ]:
normalizer = tf.keras.layers.Normalization(axis=-1)
normalizer.adapt(train_features)

Bis hierhin sollten dir die Schritte bekannt vorkommen. Jetzt aber erstellen wir unser sehr einfaches dichtes neuronales Netz mit: 
- einer Normalisierungsschicht  
- einer dichten Schicht mit 4 Knoten und der Aktivierungsfunktion gelu
- einer verborgenen Schicht mit 4 Knoten und der Aktivierungsfunktion gelu  
- einer Dropout-Schicht (Die Dropout-Schicht setzt während des Trainings bei jedem Schritt zufällig Eingabeneuronen mit der Häufigkeit rate auf 0, was hilft, Overfitting zu verhindern. Eingaben, die nicht auf 0 gesetzt werden, werden mit 1/(1 - rate) hochskaliert, sodass die Summe über alle Eingaben unverändert bleibt.)
- einer Ausgabeschicht, mit der das Testergebnis eines Autos vorhergesagt wird. Die Ausgabeschicht hat eine Sigmoid-Aktivierungsfunktion, die alle unsere Ausgaben in den Bereich zwischen 0 und 1 „quetscht“.

Anschließend kompilieren wir unser NN mit verschiedenen Hyperparametern wie:

1. Optimierer:
Die Architektur des neuronalen Netzes spielt zwar eine wichtige Rolle, wenn Informationen aus Daten gewonnen werden, aber alle (die meisten) Netze werden über Aktualisierungsregeln optimiert, die auf dem Gradienten der Verlustfunktion beruhen.
Die Aktualisierungsregeln legt der Optimierer fest. Leistung und Geschwindigkeit der Aktualisierung können sich von Optimierer zu Optimierer stark unterscheiden. Der Gradient gibt uns die Richtung der Aktualisierung vor, aber es ist noch unklar, wie groß der Schritt sein soll, den wir machen. Kurze Schritte halten uns auf Kurs, aber es kann sehr lange dauern, bis wir ein (lokales) Minimum erreichen. Große Schritte beschleunigen den Prozess, können uns aber von der richtigen Richtung abbringen.
Adam und RMSProp sind zwei sehr beliebte Optimierer, die bis heute in den meisten neuronalen Netzen eingesetzt werden. Beide aktualisieren die Variablen mit einem exponentiell abfallenden Mittelwert des Gradienten und seines Quadrats. Es gibt aber noch weitere: Wirf [einen Blick](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers) darauf.

2. Verlust
Verschiedene [Verlustfunktionen](https://www.tensorflow.org/api_docs/python/tf/keras/losses)

3. Metriken

4. und so weiter. Es gibt viele. Schau selbst in der [Dokumentation](https://www.tensorflow.org/api_docs/python/tf/keras) nach

In [ ]:
# Ein Sequential-Modell in Keras mit der folgenden Architektur definieren:
model = tf.keras.Sequential(
    [
        normalizer,
        tf.keras.layers.Dense(4, activation="gelu"),
        tf.keras.layers.Dense(4, activation="gelu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

In [ ]:
# Einen Adam-Optimierer mit einer Lernrate von 0,0005 definieren, 
# das Modell mit der binären Kreuzentropie als Verlustfunktion kompilieren 
# und Precision und Recall als Metriken aufnehmen.
opt = tf.keras.optimizers.Adam(learning_rate=0.0005)
model.compile(
    optimizer=opt,
    loss=tf.keras.losses.BinaryCrossentropy(), # für die binäre Klassifikation
    metrics=[
        tf.keras.metrics.Precision(thresholds=0.5),
        tf.keras.metrics.Recall(thresholds=0.5),
    ],
)

# Modell erstellen, damit die Zusammenfassung konkrete Eingabeformen zeigt
model.build((None, train_features.shape[1]))

In [ ]:
model.summary()

Ziel ist es, nicht bestandene Tests vorherzusagen, aber wir haben nicht sehr viele dieser positiven Beobachtungen, mit denen wir arbeiten können. Deshalb wollen wir, dass der Klassifikator die wenigen vorhandenen Beispiele stark gewichtet. Das erreichen wir, indem wir Keras über einen Parameter Gewichte für jede Klasse übergeben. Sie bewirken, dass das Modell Beispielen aus einer unterrepräsentierten Klasse „mehr Aufmerksamkeit schenkt“. Beachte aber, dass dies die Informationsmenge deines Datensatzes in keiner Weise erhöht. Sehen wir uns an, wie gut das funktioniert.

In [ ]:
# Die Skalierung mit total/2 hilft, den Verlust in einer ähnlichen Größenordnung zu halten.
# Die Summe der Gewichte aller Beispiele bleibt gleich.
total = len(train_labels)
pos = sum(train_labels)
neg = total - pos

weight_for_0 = (1 / neg) * (total / 2.0)
weight_for_1 = (1 / pos) * (total / 2.0)

class_weight = {0: weight_for_0, 1: weight_for_1}

print("Weight for class 0: {:.2f}".format(weight_for_0))
print("Weight for class 1: {:.2f}".format(weight_for_1))

In [ ]:
training = model.fit(
    train_features,
    train_labels,
    batch_size=256,
    epochs=8,
    validation_data=(val_features, val_labels),
    verbose=1,
    class_weight=class_weight,
)

Sehen wir uns unseren Recall in jeder Epoche auf dem Trainings- und dem Validierungsdatensatz an

In [ ]:
# Trainingsverlauf für den Recall zusammenfassen
plt.plot(training.history["recall"])
plt.plot(training.history["val_recall"])
plt.title("model recall")
plt.ylabel("recall")
plt.xlabel("epoch")
plt.legend(["train", "validation"], loc="upper left")
plt.grid()
plt.show()

**Aufgabe:** 
Stell den Verlust in Abhängigkeit von den Epochen dar und diskutiere das Ergebnis!

In [ ]:
# Trainingsverlauf für den Verlust zusammenfassen
plt.plot(training.history["loss"])
plt.plot(training.history["val_loss"])
plt.title("model loss")
plt.ylabel("loss")
plt.xlabel("epoch")
plt.legend(["train", "validation"], loc="upper left")
plt.grid()
plt.show()

**Erklärung**

Der Plot zeigt, dass der Trainingsverlust und der Validierungsverlust über die 8 Epochen stetig sinken (Trainingsverlust von etwa 0,73 auf 0,63, Validierungsverlust von etwa 0,68 auf 0,62). Die beiden Kurven bleiben nah beieinander, und der Validierungsverlust endet sogar etwas unter dem Trainingsverlust.

Daran sehen wir, dass das Modell **kein Overfitting zeigt**: Wäre das der Fall, würde der Validierungsverlust abflachen oder zu steigen beginnen, während der Trainingsverlust weiter sinkt. Beide Kurven zeigen in der letzten Epoche noch nach unten. Das deutet darauf hin, dass sich das Modell noch etwas verbessern könnte, wenn es ein paar Epochen länger trainiert würde. (Die genauen Zahlen ändern sich von Durchlauf zu Durchlauf leicht, weil Dropout und die Initialisierung der Gewichte zufällig sind.)

In [ ]:
y_pred = model.predict(test_features)

# Confusion Matrix darstellen
mat = confusion_matrix(test_labels, y_pred.round())
sns.heatmap(mat, square=True, annot=True, fmt="d", cbar=False)
plt.xlabel("predicted label")
plt.ylabel("true label");

In [ ]:
recall = recall_score(test_labels, y_pred.round())
round(recall, 2)

Der Recall sagt uns, welchen Anteil der Fahrzeuge, die den Abgastest nicht bestehen werden, wir tatsächlich erkennen. Um die Luftverschmutzung in Albuquerque zu verringern, soll er so hoch wie möglich sein, ohne dass wir möglichst wenige Fahrzeuge testen müssen, die den Test bestehen. Damit wir Geld sparen.

**Aufgabe:**

Ändere die Architektur und die Hyperparameter des Modells und sieh dir an, wie sich das auf den Recall auswirkt. Zum Beispiel:
- Was passiert, wenn du unserem Modell Schichten hinzufügst?
- Was passiert, wenn du die Anzahl der Knoten änderst?
- Was, wenn du die Batchgröße oder den Optimierer änderst?

In [ ]:
# Lösung: ein breiteres und tieferes Netz erstellen, zu einer größeren Lernrate und einer größeren
# Batchgröße wechseln, dann den Test-Recall mit dem oben trainierten Baseline-Modell vergleichen.

# Recall der Baseline (aus dem Modell oben) zum Vergleich behalten
baseline_recall = recall_score(test_labels, y_pred.round())

# Eine neue Normalisierungsschicht, nur an die Trainings-Features angepasst, um Data Leakage zu vermeiden
normalizer_2 = tf.keras.layers.Normalization(axis=-1)
normalizer_2.adapt(train_features)

# Mehr Schichten und mehr Knoten als im Baseline-Netz (4, 4)
model_2 = tf.keras.Sequential(
    [
        normalizer_2,
        tf.keras.layers.Dense(32, activation="gelu"),
        tf.keras.layers.Dense(16, activation="gelu"),
        tf.keras.layers.Dense(8, activation="gelu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ]
)

# Eine größere Lernrate (0,001 vs 0,0005) und derselbe Verlust/dieselben Metriken wie zuvor
model_2.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.BinaryCrossentropy(),
    metrics=[
        tf.keras.metrics.Precision(thresholds=0.5),
        tf.keras.metrics.Recall(thresholds=0.5),
    ],
)

# Eine größere Batchgröße (512 vs 256); die Klassengewichte beibehalten, um mit dem Ungleichgewicht umzugehen
training_2 = model_2.fit(
    train_features,
    train_labels,
    batch_size=512,
    epochs=8,
    validation_data=(val_features, val_labels),
    verbose=1,
    class_weight=class_weight,
)

# Test-Recall vergleichen: Baseline vs neue Architektur
y_pred_2 = model_2.predict(test_features)
recall_2 = recall_score(test_labels, y_pred_2.round())

print(f"Baseline recall:  {round(baseline_recall, 2)}")
print(f"New model recall: {round(recall_2, 2)}")

**Erklärung**

Hier haben wir das Netz breiter und tiefer gemacht (drei verborgene Schichten mit 32, 16 und 8 Knoten statt zwei Schichten mit 4), die Lernrate von 0,0005 auf 0,001 erhöht und die Batchgröße auf 512 verdoppelt. Alles andere (die Klassengewichte, der Verlust und die Metriken) blieb gleich, deshalb ist der Vergleich fair.

Der Test-Recall stieg von etwa **0,68** bei der Baseline auf etwa **0,71** beim größeren Modell. Durch die zusätzliche Kapazität kann das Netz die beiden Klassen etwas besser trennen, und bei diesem unbalancierten Problem ist der Recall die Metrik, auf die es uns ankommt: Wir wollen so viele Fahrzeuge wie möglich erkennen, die den Test nicht bestehen.

Ein paar Dinge solltest du dabei beachten:

- Der Zugewinn ist klein, und die Zahlen schwanken von Durchlauf zu Durchlauf, ein einzelner Vergleich ist also kein starker Beleg. Wenn du über mehrere Durchläufe mittelst oder einen Random Seed festlegst, bekommst du ein verlässlicheres Bild.
- Ein größeres Netz neigt stärker zu Overfitting. Beobachte den Validierungsverlust und behalte die Dropout-Schicht bei (und zieh Early Stopping in Betracht), um es unter Kontrolle zu halten.
- Der Recall allein lässt sich austricksen, indem für jedes Fahrzeug `fail` vorhergesagt wird. Prüf ihn deshalb immer zusammen mit der Precision und der Confusion Matrix.